In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf
from scipy.stats import norm

import statsmodels.graphics.tsaplots as tsp
import arviz as az
from cmdstanpy import CmdStanModel

sns.set_style('whitegrid')

# Functions

In [ ]:
# --- Grid approach ---

def get_log_likelihood(a, b, x, y):
    eta = a + b * x
    lam = np.exp(eta)
    return np.sum(y * eta - lam)

def get_percentile_interval(grid, marginal_post, confidence=0.95):
    cdf = np.cumsum(marginal_post)
    alpha = 1 - confidence
    lower_threshold = alpha / 2
    upper_threshold = 1 - (alpha / 2)
    idx_low = np.where(cdf >= lower_threshold)[0][0]
    idx_high = np.where(cdf >= upper_threshold)[0][0]
    return grid[idx_low], grid[idx_high]

def get_hpdi(grid, marginal_post, confidence=0.95):
    sorted_indices = np.argsort(marginal_post)[::-1]
    sorted_post = marginal_post[sorted_indices]
    cdf_sorted = np.cumsum(sorted_post)
    cutoff_idx = np.where(cdf_sorted >= confidence)[0][0]
    hpdi_indices = sorted_indices[:cutoff_idx + 1]
    return np.min(grid[hpdi_indices]), np.max(grid[hpdi_indices])

def get_posterior_stats(grid, marginal_post):
    p = marginal_post / np.sum(marginal_post)
    mean = np.sum(grid * p)
    variance = np.sum(((grid - mean)**2) * p)
    std_dev = np.sqrt(variance)
    return mean, std_dev


# --- MH approach ---
def plot_mcmc_diagnostics(trace, var_name, acceptance_rate):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    
    axes[0].plot(trace, color='teal', alpha=0.7, lw=0.5)
    axes[0].set_title(f'Trace Plot: {var_name}')
    axes[0].set_xlabel('Iteración')
    
    tsp.plot_acf(trace, ax=axes[1], lags=50, title=f'Autocorrelación: {var_name}')
    
    running_mean = np.cumsum(trace) / (np.arange(len(trace)) + 1)
    axes[2].plot(running_mean, color='orange')
    axes[2].set_title(f'Convergencia de la Media: {var_name}')
    
    plt.suptitle(f'Diagnóstico para {var_name} (Tasa Acep: {acceptance_rate:.1%})', fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

def plot_mcmc_joint_posterior(trace_alpha, trace_beta):
    plt.figure(figsize=(8, 5))
    
    sns.kdeplot(x=trace_alpha, y=trace_beta, fill=True, cmap='inferno', 
                thresh=0.05, levels=15, cbar=True)
    
    plt.scatter(trace_alpha[:500], trace_beta[:500], 
                color='white', s=1, alpha=0.3, label='Muestras (subset)')
    
    map_alpha = np.mean(trace_alpha)
    map_beta = np.mean(trace_beta)
    
    plt.plot(map_alpha, map_beta, 'ro', label=f'Media Post: α={map_alpha:.3f}, β={map_beta:.3f}')
    
    plt.xlabel(r'$\alpha$ (Intercepto)')
    plt.ylabel(r'$\beta$ (Pendiente)')
    plt.title('Densidad Conjunta Posterior (MCMC Samples)')
    plt.legend()
    plt.show()


def get_mcmc_stats(trace, confidence=0.95):
    """Calcula estadísticas descriptivas e intervalos para una cadena MCMC."""
    mean_val = np.mean(trace)
    std_val = np.std(trace)
    
    # Intervalo de Percentiles
    lower_p = np.percentile(trace, (1 - confidence) / 2 * 100)
    upper_p = np.percentile(trace, (1 + confidence) / 2 * 100)
    
    # HPDI (Intervalo de Máxima Densidad) para muestras
    sorted_trace = np.sort(trace)
    n_samples = len(trace)
    interval_idx = int(confidence * n_samples)
    
    # Buscamos el intervalo más corto entre todos los posibles
    width = sorted_trace[interval_idx:] - sorted_trace[:n_samples - interval_idx]
    best_start = np.argmin(width)
    hpdi_low, hpdi_high = sorted_trace[best_start], sorted_trace[best_start + interval_idx]
    
    return {
        'mean': mean_val, 'std': std_val,
        'p_low': lower_p, 'p_high': upper_p,
        'h_low': hpdi_low, 'h_high': hpdi_high
    }

def plot_parameter_comparison(trace, mle_mu, mle_se, var_label):
    stats = get_mcmc_stats(trace)
    
    plt.figure(figsize=(6, 4))
    
    # 1. Densidad Empírica (MCMC)
    sns.kdeplot(trace, label='Densidad Posterior (MCMC)', color='teal', lw=3)
    
    # Rango para las normales
    x_range = np.linspace(np.min(trace), np.max(trace), 200)
    
    # 2. Normal con Estimadores de MV (Frecuentista)
    y_mle = norm.pdf(x_range, loc=mle_mu, scale=mle_se)
    plt.plot(x_range, y_mle, label='Aprox. Normal (MLE)', color='red', linestyle='--')
    
    # 3. Normal con Estadísticos de la Posterior
    y_bayes = norm.pdf(x_range, loc=stats['mean'], scale=stats['std'])
    plt.plot(x_range, y_bayes, label='Aprox. Normal (Post Stats)', color='black', linestyle=':')
    
    # Sombrear HPDI 95%
    plt.axvspan(stats['h_low'], stats['h_high'], color='teal', alpha=0.1, label='HPDI 95%')
    
    plt.title(f'Exploración de Parámetro: {var_label}')
    plt.xlabel('Valor del Parámetro')
    plt.ylabel('Densidad')
    plt.legend()
    plt.show()
    
    print(f"--- Estadísticas para {var_label} ---")
    print(f"Media: {stats['mean']:.4f} | Desv. Est: {stats['std']:.4f}")
    print(f"Intervalo Percentiles 95%: [{stats['p_low']:.4f}, {stats['p_high']:.4f}]")
    print(f"HPDI 95%:                [{stats['h_low']:.4f}, {stats['h_high']:.4f}]")
    

# --- HMC approach ---
def plot_stan_parameter_comparison(fit, var_name, mle_mu, mle_se, grid_mu, grid_sigma, label):
    # Extraer muestras de Stan
    samples = fit.stan_variable(var_name)
    
    # Calcular estadísticas de las muestras de Stan
    hdi = az.hdi(samples, hdi_prob=0.95) # Intervalo de Máxima Densidad (HPDI)
    mu_stan = np.mean(samples)
    std_stan = np.std(samples)
    
    plt.figure(figsize=(6, 4))
    
    # 1. Densidad Empírica de Stan
    sns.kdeplot(samples, label='Posterior (Stan/HMC)', color='teal', lw=3)
    
    # Rango para las normales
    x_range = np.linspace(np.min(samples), np.max(samples), 300)
    
    # 2. Normal con Estimadores de MV (Frecuentista)
    y_mle = norm.pdf(x_range, loc=mle_mu, scale=mle_se)
    plt.plot(x_range, y_mle, label='Aprox. Normal (MLE)', color='red', linestyle='--')
    
    # 3. Normal con Estadísticos de tu Rejilla (Bayes Manual)
    y_grid = norm.pdf(x_range, loc=grid_mu, scale=grid_sigma)
    plt.plot(x_range, y_grid, label='Aprox. Normal (Grid)', color='black', linestyle=':')
    
    # Sombrear HDI 95% de Stan
    plt.axvspan(hdi[0], hdi[1], color='teal', alpha=0.1, label='HDI 95% (Stan)')
    
    plt.title(f'Comparación de Resultados: {label}')
    plt.xlabel('Valor del Parámetro')
    plt.ylabel('Densidad')
    plt.legend()
    plt.show()
    
    print(f"--- Estadísticas Stan para {label} ---")
    print(f"Media Post: {mu_stan:.4f} | Desv. Est: {std_stan:.4f}")
    print(f"95% HDI:    [{hdi[0]:.4f}, {hdi[1]:.4f}]")

def plot_density_and_normal_approx(stan_fit, var_name):
    samples = stan_fit.stan_variable(var_name)

    min_sample = np.min(samples)
    max_sample = np.max(samples)
    mean_sample = np.mean(samples) 
    std_sample = np.std(samples)

    x_plot_ = np.linspace(min_sample, max_sample, 1000)
    y_plot_ = norm.pdf(x_plot_, loc=mean_sample, scale=std_sample)

    plt.figure(figsize=(6, 4))
    sns.kdeplot(samples, label='Posterior (Stan/HMC)', color='teal', lw=3)
    sns.lineplot(x=x_plot_, y=y_plot_, label='Normal approx.', color='orange', lw=3, ls='--')
    plt.title(f'Posterior distribution of {var_name}')
    plt.xlabel('')
    plt.ylabel('')
    plt.show()    

# Carga de datos y ajuste por MV

In [ ]:
df = pd.read_csv('../data/agresti_crab_satellites_dataset.csv')


mean_W = df['W'].mean()
std_W = df['W'].std()

df['W_scaled'] = (df['W'] - mean_W) / std_W

x = df['W_scaled']
y = df['Sa'].values

model = smf.poisson('Sa ~ W_scaled', data=df).fit(disp=0)

alpha_mle = model.params['Intercept']
beta_mle = model.params['W_scaled']

alpha_mle_se = model.bse['Intercept']
beta_mle_se = model.bse['W_scaled']

print(f"MLE alpha: {alpha_mle:.3f} ± {alpha_mle_se:.3f}")
print(f"MLE alpha: {beta_mle:.3f} ± {beta_mle_se:.3f}")

# Enfoque Bayesiano - Aproximación en grid

### Grid de parámetros

In [ ]:
points = 201

alpha_min = alpha_mle - 5 * alpha_mle_se
alpha_max = alpha_mle + 5 * alpha_mle_se

beta_min = beta_mle - 5 * beta_mle_se
beta_max = beta_mle + 5 * beta_mle_se

alpha_grid = np.linspace(alpha_min, alpha_max, points)
delta_alpha = alpha_grid[1] - alpha_grid[0]
beta_grid = np.linspace(beta_min, beta_max, points)
delta_beta = beta_grid[1] - beta_grid[0]

print(f"Grid de Alpha centrado en {alpha_mle:.4f} con rango [{alpha_min:.4f}, {alpha_max:.4f}]")
print(f"Grid de Beta centrado en {beta_mle:.4f} con rango [{beta_min:.4f}, {beta_max:.4f}]")

### Cálculo de posterior y marginales

In [ ]:
A, B = np.meshgrid(alpha_grid, beta_grid)

log_likelihood_surface = np.zeros((points, points))

for i in range(points):
    for j in range(points):
        log_likelihood_surface[i, j] = get_log_likelihood(alpha_grid[j], beta_grid[i], x, y)

log_l_max = np.max(log_likelihood_surface)
posterior_unnorm = np.exp(log_likelihood_surface - log_l_max)
posterior = posterior_unnorm / np.sum(posterior_unnorm)

posterior_alpha = np.sum(posterior, axis=0)
posterior_beta = np.sum(posterior, axis=1)

### Visualizaciones

In [ ]:
plt.figure(figsize=(8, 5))
plt.contourf(A, B, posterior, cmap='inferno', levels=20)
plt.colorbar(label='Densidad Posterior')
plt.xlabel(r'$\alpha$ (Intercepto)')
plt.ylabel(r'$\beta$ (Pendiente)')
plt.title('Distribución Posterior mediante Aproximación por Rejilla')

idx = np.unravel_index(np.argmax(posterior), posterior.shape)
print(f"MAP Estimado: alpha = {alpha_grid[idx[1]]:.3f}, beta = {beta_grid[idx[0]]:.3f}")
plt.plot(alpha_grid[idx[1]], beta_grid[idx[0]], 'ro', label='MAP')
plt.legend()
plt.show()

In [ ]:
alpha_low_p, alpha_high_p = get_percentile_interval(alpha_grid, posterior_alpha, 0.95)
alpha_low_h, alpha_high_h = get_hpdi(alpha_grid, posterior_alpha, 0.95)
mu_alpha, sigma_alpha = get_posterior_stats(alpha_grid, posterior_alpha)

densidad_mle = norm.pdf(alpha_grid, loc=alpha_mle, scale=alpha_mle_se)
densidad_bayes = norm.pdf(alpha_grid, loc=mu_alpha, scale=sigma_alpha)

plt.figure(figsize=(6, 4))
sns.lineplot(x=alpha_grid, y=posterior_alpha/delta_alpha, label='Posterior')
sns.lineplot(x=alpha_grid, y=densidad_mle, label='Densidad MLE', linestyle='--')
sns.lineplot(x=alpha_grid, y=densidad_bayes, label='Densidad Bayesiana', linestyle=':')
plt.title('Distribución Posterior Marginal de $\\alpha$')
plt.legend()
plt.show()

print(f"Intervalo Percentiles: [{alpha_low_p:.3f}, {alpha_high_p:.3f}]")
print(f"Intervalo HPDI:        [{alpha_low_h:.3f}, {alpha_high_h:.3f}]")
print(f"alpha: {mu_alpha:.4f} ± {sigma_alpha:.4f}")

In [ ]:
beta_low_p, beta_high_p = get_percentile_interval(beta_grid, posterior_beta, 0.95)
beta_low_h, beta_high_h = get_hpdi(beta_grid, posterior_beta, 0.95)
mu_beta, sigma_beta = get_posterior_stats(beta_grid, posterior_beta)

densidad_mle = norm.pdf(beta_grid, loc=beta_mle, scale=beta_mle_se)
densidad_bayes = norm.pdf(beta_grid, loc=mu_beta, scale=sigma_beta)

plt.figure(figsize=(6, 4))
sns.lineplot(x=beta_grid, y=posterior_beta/delta_beta, label='Posterior')
sns.lineplot(x=beta_grid, y=densidad_mle, label='Densidad MLE', linestyle='--')
sns.lineplot(x=beta_grid, y=densidad_bayes, label='Densidad Bayesiana', linestyle=':')
plt.title('Distribución Posterior Marginal de $\\beta$')
plt.show()

print(f"Intervalo Percentiles: [{beta_low_p:.3f}, {beta_high_p:.3f}]")
print(f"Intervalo HPDI:        [{beta_low_h:.3f}, {beta_high_h:.3f}]")
print(f"Beta: {mu_beta:.4f} ± {sigma_beta:.4f}")

# Metropolis-Hastings

In [ ]:
n_iterations = 20000
burn_in = 5000

alpha_curr = alpha_mle 
beta_curr = beta_mle

trace_alpha = np.zeros(n_iterations)
trace_beta = np.zeros(n_iterations)
accepted = 0

proposal_width_a = alpha_mle_se * 2
proposal_width_b = beta_mle_se * 2

for i in range(n_iterations):

    alpha_prop = np.random.normal(alpha_curr, proposal_width_a)
    beta_prop = np.random.normal(beta_curr, proposal_width_b)
    
    lp_curr = get_log_likelihood(alpha_curr, beta_curr, x, y)
    lp_prop = get_log_likelihood(alpha_prop, beta_prop, x, y)
    
    log_r = lp_prop - lp_curr
    
    if np.log(np.random.uniform(0, 1)) < log_r:
        alpha_curr = alpha_prop
        beta_curr = beta_prop
        accepted += 1
        
    trace_alpha[i] = alpha_curr
    trace_beta[i] = beta_curr

trace_alpha = trace_alpha[burn_in:]
trace_beta = trace_beta[burn_in:]
acceptance_rate = accepted / n_iterations

print(f"Tasa de aceptación: {acceptance_rate:.2%}")
print(f"Alpha (Posterior Mean): {np.mean(trace_alpha):.4f}")
print(f"Beta  (Posterior Mean): {np.mean(trace_beta):.4f}")

### Dianósticos de la cadena

In [ ]:
plot_mcmc_diagnostics(trace_alpha, 'Alpha (Intercepto)', acceptance_rate)
plot_mcmc_diagnostics(trace_beta, 'Beta (Ancho)', acceptance_rate)

### Análisis de valores simulados

In [ ]:
plot_mcmc_joint_posterior(trace_alpha, trace_beta)

In [ ]:
plot_parameter_comparison(trace_alpha, alpha_mle, alpha_mle_se, r"$\alpha$ (Intercepto)")
plot_parameter_comparison(trace_beta, beta_mle, beta_mle_se, r"$\beta$ (Pendiente)")

# Stan

In [ ]:
stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values,
    'y': df['Sa'].values
}

model_stan = CmdStanModel(stan_file='../models/poisson_model.stan')
fit = model_stan.sample(data=stan_data, chains=4, iter_sampling=2000)

In [ ]:
display(fit.summary())

alpha_samples = fit.stan_variable('alpha')
beta_samples = fit.stan_variable('beta')

az.plot_trace(fit, var_names=['alpha', 'beta'], compact=False)
plt.tight_layout()
plt.show()

# 4. Gráfica de Autocorrelación (Stan suele tener mucha menos que MH)
az.plot_autocorr(fit, var_names=['alpha', 'beta'], combined=True)
plt.show()

In [ ]:
plot_mcmc_joint_posterior(alpha_samples, beta_samples)

In [ ]:
plot_stan_parameter_comparison(fit, 'alpha', alpha_mle, alpha_mle_se, mu_alpha, sigma_alpha, r"$\alpha$ (Intercepto)")
plot_stan_parameter_comparison(fit, 'beta', beta_mle, beta_mle_se, mu_beta, sigma_beta, r"$\beta$ (Ancho)")

# Guardar InferenceData

Convierte los fits de cmdstanpy a `az.InferenceData` y los guarda en `outputs/`.
Incluye `posterior_predictive` (y_rep) y `log_likelihood` (log_lik) generados por Stan.

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

idata_poisson = az.from_cmdstanpy(
    fit,
    observed_data={'y': stan_data['y']},
    posterior_predictive='y_rep',
    log_likelihood='log_lik'
)
az.to_netcdf(idata_poisson, '../outputs/pois_idata.nc')
print("Guardado: ../outputs/pois_idata.nc")

# Negative binomial model

In [ ]:
stan_data = {
    'N': len(df),
    'x': df['W_scaled'].values,
    'y': df['Sa'].values
}

def get_inits():
    return {
        "alpha": 1.0, 
        "beta": 0.4, 
        "phi": 1.0
    }

model_nb = CmdStanModel(stan_file='../models/negative_binomial_model.stan')
fit_nb = model_nb.sample(data=stan_data, chains=4, iter_sampling=2000, inits=get_inits())

In [ ]:
display(fit_nb.summary())

az.plot_trace(fit_nb, var_names=['alpha', 'beta','alpha_sm'], compact=False)
plt.tight_layout()
plt.show()


# 4. Gráfica de Autocorrelación (Stan suele tener mucha menos que MH)
az.plot_autocorr(fit_nb, var_names=['alpha', 'beta','alpha_sm'], combined=True)
plt.show()

In [ ]:
alpha_nb_samples = fit_nb.stan_variable('alpha')
beta_nb_samples = fit_nb.stan_variable('beta')
alpha_sm_nb_samples = fit_nb.stan_variable('alpha_sm')


mean_alpha_nb = np.mean(alpha_nb_samples)
std_alpha_nb = np.std(alpha_nb_samples)

mean_beta_nb = np.mean(beta_nb_samples)
mean_beta_nb = np.std(beta_nb_samples)

mean_alpha_sm_nb = np.mean(alpha_sm_nb_samples)
std_alpha_sm_nb = np.std(alpha_sm_nb_samples)

In [ ]:
plot_mcmc_joint_posterior(alpha_nb_samples, beta_nb_samples)

In [ ]:
plot_density_and_normal_approx(fit_nb, var_name='alpha')
plot_density_and_normal_approx(fit_nb, var_name='beta')
plot_density_and_normal_approx(fit_nb, var_name='alpha_sm')

In [ ]:
idata_nb = az.from_cmdstanpy(
    fit_nb,
    observed_data={'y': stan_data['y']},
    posterior_predictive='y_rep',
    log_likelihood='log_lik'
)
az.to_netcdf(idata_nb, '../outputs/neg_idata.nc')
print("Guardado: ../outputs/neg_idata.nc")